In [35]:
import torch
from transformers import GPT2Tokenizer, GPT2Model

torch.set_grad_enabled(False)

model = GPT2Model.from_pretrained("gpt2")
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model.eval()



Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2Model(
  (wte): Embedding(50257, 768)
  (wpe): Embedding(1024, 768)
  (drop): Dropout(p=0.1, inplace=False)
  (h): ModuleList(
    (0-11): 12 x GPT2Block(
      (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (attn): GPT2Attention(
        (c_attn): Conv1D(nf=2304, nx=768)
        (c_proj): Conv1D(nf=768, nx=768)
        (attn_dropout): Dropout(p=0.1, inplace=False)
        (resid_dropout): Dropout(p=0.1, inplace=False)
      )
      (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (mlp): GPT2MLP(
        (c_fc): Conv1D(nf=3072, nx=768)
        (c_proj): Conv1D(nf=768, nx=3072)
        (act): NewGELUActivation()
        (dropout): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
)

In [24]:
#Tokenizador

texts = ["Hello, good morning"]

encoded = tokenizer(texts, return_tensors="pt")
print(encoded)

ids_input = encoded["input_ids"]

num_tokens = ids_input.shape[1]

print("tokens: ", tokenizer.convert_ids_to_tokens(ids_input[0]))


{'input_ids': tensor([[15496,    11,   922,  3329]]), 'attention_mask': tensor([[1, 1, 1, 1]])}
tokens:  ['Hello', ',', 'Ġgood', 'Ġmorning']


In [25]:
#Cálculo de wpe y wte

wpe = model.wpe.weight
wte = model.wte.weight

positions = torch.arange(num_tokens)

embeddings = wpe[positions] + wte[ids_input]

print(embeddings)


tensor([[[-0.0875, -0.3301,  0.0152,  ...,  0.0285, -0.0014,  0.0068],
         [ 0.0355, -0.0567, -0.0626,  ...,  0.0619, -0.0195, -0.0601],
         [-0.0796, -0.1353,  0.0913,  ..., -0.1339, -0.0702, -0.1327],
         [ 0.0633,  0.0460,  0.1728,  ...,  0.0879,  0.0270, -0.1788]]])


In [26]:
#Huggingface embeddings results

salida_modelo = model(ids_input, output_hidden_states=True)

embeddings_modelo = salida_modelo.hidden_states[0]

print(embeddings_modelo)

tensor([[[-0.0875, -0.3301,  0.0152,  ...,  0.0285, -0.0014,  0.0068],
         [ 0.0355, -0.0567, -0.0626,  ...,  0.0619, -0.0195, -0.0601],
         [-0.0796, -0.1353,  0.0913,  ..., -0.1339, -0.0702, -0.1327],
         [ 0.0633,  0.0460,  0.1728,  ...,  0.0879,  0.0270, -0.1788]]])


In [27]:
def comparar(tensor_1, tensor_2):
    diferencias = (tensor_1 - tensor_2).abs()
    diferencia_max = diferencias.max().item()

    torch.testing.assert_close(tensor_1, tensor_2)

    print(f"Diferencia máxima: {diferencia_max}")


comparar(embeddings, embeddings_modelo)

Diferencia máxima: 0.0


In [28]:
#LayerNorm

def layernorm(tensor,weight,bias):
    mean = torch.mean(tensor, dim=-1, keepdim=True)
    eps = 1e-5
    variance = torch.var(tensor, dim=-1, keepdim=True, correction=0)

    output = ((tensor-mean)/torch.sqrt(variance+eps))* weight + bias
    return output


In [29]:
mi_resultado_layernorm = layernorm(embeddings,model.h[0].ln_1.weight,model.h[0].ln_1.bias)

In [30]:
#Huggingface h[0] ln_1 results

resultado_oficial = model.h[0].ln_1(embeddings_modelo)

comparar(mi_resultado_layernorm, resultado_oficial)

Diferencia máxima: 5.960464477539063e-08


In [43]:
#MLP Implementation
import torch.nn.functional as fu

def mlp(tensor, weight_1, bias_1, weight_2, bias_2):
    inter = tensor @ weight_1 + bias_1
    inter_2 = fu.gelu(inter, approximate="tanh")
    output = inter_2 @ weight_2 + bias_2

    return output



In [44]:
mi_resultado_mlp = mlp(embeddings, model.h[0].mlp.c_fc.weight, model.h[0].mlp.c_fc.bias, model.h[0].mlp.c_proj.weight, model.h[0].mlp.c_proj.bias)
resultado_oficial_2 = model.h[0].mlp(embeddings)

comparar(mi_resultado_mlp, resultado_oficial_2)


Diferencia máxima: 3.0517578125e-05
